# 02 — Agent Pipeline Inspection

**Purpose**: Run this notebook to trace a complete agent pipeline run step by step.  
It runs `IngestionRunner` against an AMFI NAV URL with `max_pages=2` using SQLite,  
then inspects every DB table to confirm the pipeline wrote records correctly.

**When to run**: After any change to runner.py, discovery.py, parser/, validate.py, or db.py.  
**What it does NOT do**: Make unlimited network calls; write to production PostgreSQL.

In [ ]:
# Setup
import sys
import logging
import tempfile
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s'
)
logger = logging.getLogger('pipeline_inspection')

ROOT = Path('.').resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mutual_fund_ingestion.agent.config import AgentConfig
from mutual_fund_ingestion.agent.db import create_tables
from mutual_fund_ingestion.agent.runner import IngestionRunner

print(f'[SETUP] Root: {ROOT}')
print('✓ Setup complete')

## Database Initialization

In [ ]:
print('[STAGE] Initializing SQLite database ...')
tmp_dir = tempfile.mkdtemp(prefix='pipeline_inspect_')
db_path = Path(tmp_dir) / 'inspect.db'
db_url = f'sqlite:///{db_path}'

from sqlalchemy import create_engine, inspect as sa_inspect
create_tables(db_url)
engine = create_engine(db_url)

inspector = sa_inspect(engine)
tables = inspector.get_table_names()
print(f'Tables created: {len(tables)}')
for t in sorted(tables):
    print(f'  - {t}')
assert len(tables) >= 17, f'Expected 17 tables, got {len(tables)}'
print('✓ DB initialized with all 17 tables')

## Config

In [ ]:
print('[STAGE] Building AgentConfig ...')
TEST_URL = 'https://www.amfiindia.com/spages/NAVAll.txt'

config = AgentConfig(
    task_url=TEST_URL,
    database_url=db_url,
    max_pages=2,
    dry_run=False,
    use_vlm=False,
    keep_raw_files=False,
)
print(f'task_url: {config.task_url}')
print(f'max_pages: {config.max_pages}')
print(f'database_url: sqlite:///<tmp>')
print('✓ Config ready')

## Run: Bounded Crawl (max_pages=2)

> **Note**: This makes a real HTTP request to AMFI. If the network is unavailable, this cell will timeout gracefully.

In [ ]:
print('[STAGE] Running IngestionRunner (max_pages=2) ...')
try:
    runner = IngestionRunner(config)
    run_result = runner.run()
    print(f'Run result keys: {list(run_result.keys()) if run_result else "None"}')
    print('✓ Run complete')
except Exception as e:
    print(f'⚠️  Run raised {type(e).__name__}: {e}')
    print('   (Network errors are expected in offline environments)')
    run_result = None

## Inspect: ingestion_runs

In [ ]:
print('[STAGE] Inspecting ingestion_runs table ...')
from sqlalchemy import text
with engine.connect() as conn:
    rows = conn.execute(text('SELECT run_id, status, start_time FROM ingestion_runs LIMIT 5')).fetchall()
print(f'ingestion_runs rows: {len(rows)}')
for row in rows:
    print(f'  run_id={row[0]}, status={row[1]}, start={row[2]}')
print('✓ ingestion_runs inspected')

## Inspect: source_pages

In [ ]:
print('[STAGE] Inspecting source_pages ...')
with engine.connect() as conn:
    rows = conn.execute(text('SELECT url, status_code, relevance_score FROM source_pages LIMIT 5')).fetchall()
print(f'source_pages rows: {len(rows)}')
for row in rows:
    print(f'  url={str(row[0])[:60]}, status={row[1]}, relevance={row[2]}')
print('✓ source_pages inspected')

## Inspect: discovered_links, dataset_candidates, raw_artifacts

In [ ]:
print('[STAGE] Inspecting pipeline tables ...')
table_checks = ['discovered_links', 'dataset_candidates', 'raw_artifacts', 'staging_rows', 'validation_results', 'quarantine_rows']
with engine.connect() as conn:
    for table in table_checks:
        try:
            count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
            print(f'  {table}: {count} rows')
        except Exception as e:
            print(f'  {table}: ERROR — {e}')
print('✓ Pipeline tables inspected')

## Inspect: Canonical Tables

In [ ]:
print('[STAGE] Inspecting canonical tables ...')
canonical = ['amcs', 'schemes', 'nav_history', 'portfolio_holdings']
with engine.connect() as conn:
    for table in canonical:
        try:
            count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
            print(f'  {table}: {count} rows')
        except Exception as e:
            print(f'  {table}: ERROR — {e}')
print('✓ Canonical tables inspected')

## Edge Case: Invalid URL

In [ ]:
print('[STAGE] Testing invalid URL handling ...')
bad_config = AgentConfig(
    task_url='https://this-does-not-exist.invalid/nav.txt',
    database_url=db_url,
    max_pages=1,
    dry_run=False,
    use_vlm=False,
    keep_raw_files=False,
)
try:
    bad_runner = IngestionRunner(bad_config)
    bad_result = bad_runner.run()
    print('Run completed (may have logged errors internally)')
    print(f'Result: {bad_result}')
except Exception as e:
    print(f'Run raised {type(e).__name__}: {e}')
    print('(Expected — network failure handled)')
print('✓ Invalid URL test complete')

## Assertions

In [ ]:
print('[STAGE] Running assertions ...')
# DB must have all 17 tables
assert len(tables) >= 17, f'Expected 17 tables, got {len(tables)}'
# ingestion_runs must have at least 1 row (the failed run from bad URL also creates a row)
with engine.connect() as conn:
    run_count = conn.execute(text('SELECT COUNT(*) FROM ingestion_runs')).scalar()
assert run_count >= 1, f'Expected ≥1 ingestion run, got {run_count}'
print(f'ingestion_runs: {run_count} rows ✓')
print('✓ All assertions passed')

## Debugging Notes

| Stage | Where to look |
|---|---|
| Run fails to start | `agent/config.py` — check AgentConfig fields |
| No source_pages rows | `agent/discovery.py` — BFS not crawling |
| No dataset_candidates | `agent/discovery.py` — `classify_dataset()` not matching URL |
| No raw_artifacts | `agent/extract.py` — download failing or file type rejected |
| Staging rows but no canonical | `agent/validate.py` — all records quarantined |
| Quarantine rows | `agent/validate.py` — check `reason_code` field |
| 17 tables missing | `agent/db.py` — `create_tables()` incomplete |

## Summary

In [ ]:
print('=== PIPELINE INSPECTION SUMMARY ===')
with engine.connect() as conn:
    for table in sorted(tables):
        try:
            count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
            icon = '🟢' if count > 0 else '⚪'
            print(f'  {icon} {table}: {count} rows')
        except Exception as e:
            print(f'  🔴 {table}: ERROR')
print()
print('DB path:', db_path)
print('====================================')